In [1]:
!pip install qdrant-client open-clip-torch pillow torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/jnDhruv/multi-modal-fashion-ecom
!cd /kaggle/working/multi-modal-fashion-ecom && git pull

Cloning into 'multi-modal-fashion-ecom'...
remote: Enumerating objects: 422, done.
remote: Counting objects: 100% (422/422), done.
remote: Compressing objects: 100% (367/367), done.
remote: Total 422 (delta 63), reused 351 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (422/422), 6.13 MiB | 21.93 MiB/s, done.
Resolving deltas: 100% (63/63), done.
Already up to date.


In [11]:
from kaggle_secrets import UserSecretsClient
from qdrant_client import QdrantClient

# Fetch your secret key securely from Kaggle
user_secrets = UserSecretsClient()
qdrant_key = user_secrets.get_secret("api_key")

In [4]:
import sys 
import os 
abs_repo = "/kaggle/working/multi-modal-fashion-ecom/backend/app"

if abs_repo not in sys.path :
    sys.path.append(abs_repo)
if os.path.exists(os.path.join(abs_repo, "search_engine.py")):
    print("File found")
else :
    print("File not found")

File found


In [5]:
import torch 
import open_clip
from PIL import Image
from qdrant_client import QdrantClient
from qdrant_client.http import models
from typing import List, Dict, Any, Optional
from sentence_transformers import CrossEncoder

In [6]:
import sys
import torch
import open_clip
from PIL import Image
from typing import List, Dict, Any, Optional
from qdrant_client import QdrantClient
from qdrant_client.http import models
from sentence_transformers import CrossEncoder

class SearchEngine:
    def __init__(self, qurl: str, api: Optional[str] = None):
     
  
        try:
            self.client = QdrantClient(url=qurl, api_key=api)
        except Exception:
            sys.exit("Error connecting to database")

       
        try:
            collections_response = self.client.get_collections()
            if collections_response.collections:
                self.collection_name = collections_response.collections[0].name
                print(f"Connected to Qdrant. Automatically using active collection: '{self.collection_name}'")
            else:
                sys.exit("Error: No active collections found on this Qdrant server instance.")
        except Exception as e:
            sys.exit(f"Error fetching collection lists: {e}")
        
       
        print("Configuring Marqo Fashion-CLIP encoding layers...")
        self.M = "hf-hub:Marqo/marqo-fashionCLIP"
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(self.M)
        self.tokenizer = open_clip.get_tokenizer(self.M)
        self.model.eval()
        
        print("Preloading cross-encoder ranking optimization architectures...")
        self.cross_encoder_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    def generate_query_embedding(self, text: Optional[str] = None, image: Optional[Image.Image] = None) -> List[float]:
    
        feature_t = None
        feature_i = None 

        with torch.no_grad():
            if text: 
                tokens = self.tokenizer([text])
                feature_t = self.model.encode_text(tokens)
                feature_t /= feature_t.norm(dim=-1, keepdim=True)
            if image:
                tensor = self.preprocess(image).unsqueeze(0)
                feature_i = self.model.encode_image(tensor)  # Fixed typo: was model.encode_text(tensor)
                feature_i /= feature_i.norm(dim=-1, keepdim=True)
                
            if feature_t is not None and feature_i is not None:
                blend = feature_t + feature_i
                blend /= blend.norm(dim=-1, keepdim=True)
                return blend.squeeze(0).tolist()
            elif feature_t is not None:
                return feature_t.squeeze(0).tolist()
            elif feature_i is not None:
                return feature_i.squeeze(0).tolist()
            else:
                raise ValueError("You must provide at least a text query or an image query!")

    def _build_qdrant_filters(self, dic: Optional[Dict[str, Any]] = None) -> Optional[models.Filter]:
    
        if not dic:
            return None
            
        must_clauses = []
        fields = [
            "brand_name", "gender", "master_category", "sub_category", 
            "article_type", "base_colour", "season", "usage", "year",
            "pattern", "fabric", "sleeve_length", "occasion", "fit", "neck", "length"
        ]

       
        for field in fields:
            if field in dic and dic[field] is not None: 
                val = dic[field]
                if isinstance(val, list):
                    must_clauses.append(
                        models.FieldCondition(
                            key=field,
                            match=models.MatchAny(any=val)
                        )
                    )
                else: 
                    must_clauses.append(
                        models.FieldCondition(
                            key=field,
                            match=models.MatchValue(value=val)
                        )
                    )

        if "min_price" in dic and dic["min_price"] is not None:
            must_clauses.append(
                models.FieldCondition(
                    key="discounted_price",
                    range=models.Range(gte=float(dic["min_price"]))
                )
            )
    
        if "max_price" in dic and dic["max_price"] is not None:
            must_clauses.append(
                models.FieldCondition(
                    key="discounted_price",
                    range=models.Range(lte=float(dic["max_price"]))
                )
            )

        return models.Filter(must=must_clauses) if must_clauses else None
    
    def hybrid_search(
        self, 
        dense_vector: List[float], 
        text_query: Optional[str] = None,
        hard_filters: Optional[models.Filter] = None,
        limit_can: int = 50
    ) -> List[Any]:
      
        hnsw = models.Prefetch(
            query=dense_vector,
            using="dense",
            filter=hard_filters,
            limit=limit_can
        )
        
        bm25 = models.Prefetch(
            query=models.Document(
                text=text_query,
                model="Qdrant/bm25"
            ),
            using="sparse",
            filter=hard_filters,
            limit=limit_can
        )
        
        response = self.client.query_points(
            collection_name=self.collection_name,
            prefetch=[hnsw, bm25],
            query=models.FusionQuery(fusion=models.Fusion.RRF), 
            query_filter=hard_filters,
            limit=limit_can,
            with_payload=True
        )
        return response.points
       
    def execute_retrieval(
        self, 
        dense_vector: List[float], 
        text_query: Optional[str] = None,
        filter_dict: Optional[Dict[str, Any]] = None,
        limit_candidates: int = 50
    ) -> List[Any]:
       
        hard_filters = self._build_qdrant_filters(filter_dict)
        
        if text_query:
            return self.hybrid_search(
                dense_vector=dense_vector, 
                text_query=text_query,
                hard_filters=hard_filters,
                limit_can=limit_candidates
            )
        else:
            response = self.client.query_points(
                collection_name=self.collection_name,
                query=dense_vector,
                using="dense",
                query_filter=hard_filters,
                limit=limit_candidates,
                with_payload=True
            )
            return response.points
            
    def precision_rerank(self, query_text: Optional[str] = None, items_found: Optional[List[Any]] = None) -> List[int]:
       
        if not query_text or not items_found:
            return [int(i.id) for i in items_found] if items_found else []

        pairs = []
        valid_pids = []
        
        for i in items_found:
            payload = i.payload or {}
            title = payload.get("product_display_name", "")
            desc = payload.get("description", "")
            content = f"{title} {desc}".strip()
            
            if content:
                pairs.append([query_text, content])
                valid_pids.append(int(i.id))
                
        if not pairs:
            return [int(p.id) for p in items_found]
            
        scores = self.cross_encoder_model.predict(pairs)
        reranked_pairs = sorted(zip(valid_pids, scores), key=lambda x: x[1], reverse=True)
        return [pid for pid, score in reranked_pairs]

    def discover_fashion(
        self, 
        text_query: Optional[str] = None, 
        image_query: Optional[Image.Image] = None,
        filters: Optional[Dict[str, Any]] = None,
        apply_rerank: bool = False,
        top_k: int = 10
    ) -> List[int]:
       
        dense_vector = self.generate_query_embedding(text=text_query, image=image_query)

        limit_candidates = 30 if apply_rerank else top_k
        points = self.execute_retrieval(
            dense_vector=dense_vector,
            text_query=text_query,
            filter_dict=filters,
            limit_candidates=limit_candidates
        )

        if apply_rerank and text_query and points:
            final_ordered_ids = self.precision_rerank(query_text=text_query, items_found=points)
        else:
            final_ordered_ids = [int(p.id) for p in points]

        return final_ordered_ids[:top_k]


In [12]:
QDRANT_CLUSTER_URL = "https://ebace295-dfd7-4b27-b71f-e4311b5ec3fc.eu-central-1-0.aws.cloud.qdrant.io" 

engine = SearchEngine(qurl=QDRANT_CLUSTER_URL, api=qdrant_key)

search_term = "women black jeans"
hard_filters = {
   
}

try:
    print(f"Searching for: '{search_term}'...")
    matched_product_ids = engine.discover_fashion(
        text_query=search_term,
        filters=hard_filters,
        apply_rerank=True,
        top_k=15
    )
    print("Returned Product ID Matches:", matched_product_ids)
except Exception as e:
    print(f"Pipeline execution error: {e}")


Connected to Qdrant. Automatically using active collection: 'products'
Configuring Marqo Fashion-CLIP encoding layers...


open_clip_config.json:   0%|          | 0.00/532 [00:00<?, ?B/s]

open_clip_model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Preloading cross-encoder ranking optimization architectures...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Searching for: 'women black jeans'...
Returned Product ID Matches: [52528, 32559, 41353, 57071, 48494, 26994, 51605, 51596, 51607, 51601, 43318, 51611, 50951, 51616, 50948]


In [13]:
#display purposes
from IPython.display import display, HTML

print(f"--- Fetching metadata payloads for matching IDs: {matched_product_ids} ---")

# 1. Retrieve the full payload objects directly from your live Qdrant cluster points
points_data = engine.client.retrieve(
    collection_name="products",
    ids=matched_product_ids,
    with_payload=True
)

# 2. Put them into a dictionary keyed by ID so we can sort them back to your exact ranking order
points_dict = {p.id: p.payload for p in points_data}

# 3. Build an elegant, scannable HTML grid to display inside your notebook
html_content = """
<div style="font-family: Arial, sans-serif; max-width: 900px; margin: 0 auto;">
    <h2 style="color: #2c3e50; border-bottom: 2px solid #ecf0f1; padding-bottom: 10px; text-align: center;">
        🛍️ Fashion Discovery Engine — Search Results
    </h2>
"""

for rank, pid in enumerate(matched_product_ids, 1):
    payload = points_dict.get(pid)
    
    if not payload:
        continue
    
    # Extract metadata fields matching your teammate's schema setup exactly
    name = payload.get("product_display_name", "Unknown Item")
    brand = payload.get("brand_name", "Generic")
    colour = payload.get("base_colour", "N/A")
    price = payload.get("price", 0)
    disc_price = payload.get("discounted_price", price)
    img_url = payload.get("image_url", "https://placeholder.com")
    details = payload.get("description", "No detailed description available.")
    
    # Calculate discount tag if applicable
    discount_text = f"<span style='color: #27ae60; font-weight: bold;'>INR {disc_price}</span> <span style='text-decoration: line-through; color: #7f8c8d; font-size: 0.9em; margin-left: 5px;'>INR {price}</span>" if disc_price < price else f"<span style='font-weight: bold;'>INR {price}</span>"

    # Append item layout row
    html_content += f"""
    <div style="display: flex; background: #ffffff; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); margin: 20px 0; padding: 15px; border: 1px solid #e2e8f0; align-items: center;">
        <div style="flex: 0 0 40px; font-size: 1.5em; font-weight: bold; color: #7f8c8d; text-align: center; margin-right: 15px;">
            #{rank}
        </div>
        <div style="flex: 0 0 120px; text-align: center; margin-right: 20px;">
            <img src="{img_url}" alt="{name}" style="max-width: 100%; max-height: 140px; border-radius: 4px; object-fit: contain; border: 1px solid #f1f5f9;"/>
        </div>
        <div style="flex: 1;">
            <h3 style="margin: 0 0 5px 0; color: #2d3748; font-size: 1.2em;">{name}</h3>
            <p style="margin: 0 0 8px 0; font-size: 0.95em; color: #4a5568;">
                <strong>Brand:</strong> {brand} | <strong>Colour:</strong> {colour}
            </p>
            <p style="margin: 0 0 10px 0; font-size: 0.9em; color: #718096; line-height: 1.4;">
                <em>{details}</em>
            </p>
            <div style="font-size: 1.1em;">
                {discount_text}
            </div>
            <p style="margin: 5px 0 0 0; font-size: 0.8em; color: #a0aec0;">Product ID: {pid}</p>
        </div>
    </div>
    """

html_content += "</div>"

# 4. Physically inject and display the UI card elements straight onto the Kaggle dashboard screen
display(HTML(html_content))


--- Fetching metadata payloads for matching IDs: [52528, 32559, 41353, 57071, 48494, 26994, 51605, 51596, 51607, 51601, 43318, 51611, 50951, 51616, 50948] ---
